####DAY 12 (03/03/26) – Cost Optimization Basics
####🏗️ Architecture & Strategy
Welcome to Day 12! Today, we shift our mindset from "making it work" to "making it profitable."

Imagine scaling a civic issue reporting platform for an entire state government: if every citizen's report triggers a full, unoptimized database scan, the cloud compute bill will bankrupt the project before it even launches. In the cloud, compute time directly equals money.

####Our Senior-Level Strategy:

* **Understanding Lazy Evaluation**: Apache Spark is "lazy." It only executes transformations when you explicitly call an "Action" (like `.count(), .show()`, or `.write()`). A common junior mistake is calling multiple actions for debugging, which forces Spark to re-read the massive source data from disk every single time.

* **The Anti-Pattern (Cell 1)**: We will deliberately write a highly inefficient PySpark script that triggers multiple unnecessary actions to analyze job runtime.

* **The Optimized Pattern (Cell 2)**: We will rewrite that exact same logic to reduce unnecessary actions, condensing multiple disk reads into a single, highly optimized pass over the data.

* **Strategic Cost Documentation (Cell 3)**: We will document enterprise-grade cost-saving ideas, comparing Interactive Clusters vs. Job Clusters.

####The Junior Anti-Pattern (Unnecessary Actions)
Let's look at how not to write PySpark. This code calculates the total counts for purchases, carts, and views by triggering three separate `.count()` actions.

In [0]:
import time
from pyspark.sql import functions as F

catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("🚨 RUNNING UNOPTIMIZED PIPELINE (Multiple Actions)...")
start_time = time.time()

# Load the massive 56M row table
df_events = spark.table("events_delta_managed")

# ANTI-PATTERN: Triggering an action (.count) for every single metric.
# Because Spark is lazy, it has to scan the 56M rows from disk THREE separate times!
purchase_count = df_events.filter(F.col("event_type") == "purchase").count()
cart_count = df_events.filter(F.col("event_type") == "cart").count()
view_count = df_events.filter(F.col("event_type") == "view").count()

# ANTI-PATTERN: Using .show() triggers yet another computation job just to print to the console
print(f"Purchases: {purchase_count}, Carts: {cart_count}, Views: {view_count}")

unoptimized_time = time.time() - start_time
print(f"❌ Unoptimized Execution Time: {unoptimized_time:.2f} seconds")

####The Senior Pattern (Optimized Execution)
Now, let's rewrite this using a single action. We will use a groupBy transformation to calculate everything at once, meaning Spark only has to read the 56 million rows from the disk one time.

In [0]:
print("🚀 RUNNING OPTIMIZED PIPELINE (Single Action)...")
start_time = time.time()

# 1. Chain Transformations (No actions triggered yet!)
# We tell Spark exactly what we want, letting the Catalyst Optimizer build the most efficient plan.
optimized_summary_df = (
    spark.table("events_delta_managed")
    .groupBy("event_type")
    .agg(F.count("*").alias("total_events"))
)

# 2. Trigger a Single Action
# display() is heavily optimized for Databricks. It triggers the computation ONCE 
# and natively limits the UI rendering without overloading the driver node.
display(optimized_summary_df)

optimized_time = time.time() - start_time

print(f"✅ Optimized Execution Time: {optimized_time:.2f} seconds")

# Calculate the actual monetary/time savings
if optimized_time < unoptimized_time:
    speedup = unoptimized_time / optimized_time
    print(f"💰 Cost Optimization: Pipeline ran {speedup:.1f}x faster, directly reducing compute costs!")

####Enterprise Cost-Saving Documentation
In a real portfolio, having a Markdown cell dedicated to FinOps (Financial Operations) shows profound maturity.

### 💸 Databricks Cost Optimization Architecture

To ensure our AI and Data Engineering pipelines remain financially viable at scale, we must implement the following Databricks FinOps best practices:

1. **Job Clusters vs. All-Purpose Clusters:**
   * **The Trap:** Running scheduled production pipelines on "All-Purpose" (Interactive) compute.
   * **The Fix:** Production pipelines must strictly run on **Job Compute**. Job clusters cost significantly less (often 50% cheaper per DBU) because they terminate immediately after the task finishes, whereas All-Purpose clusters are billed at a premium for interactive UI features.
2. **Photon Engine Enablement:**
   * For heavy aggregation and join workloads, enabling the **Photon** engine (Databricks' native C++ execution engine) speeds up query execution drastically. While Photon DBUs cost slightly more per hour, the job finishes so much faster that the overall total cost of the run decreases.
3. **Optimized Auto-Scaling:**
   * Never hardcode a static cluster size of 20 workers for a pipeline that only needs 20 workers for 5 minutes and 2 workers for the rest of the hour. Enable cluster auto-scaling with aggressive scale-down rules to release unused instances back to the cloud provider.
4. **Data Layout (Z-Ordering & Partitioning):**
   * Storing data efficiently using `OPTIMIZE` and `ZORDER` drastically reduces the amount of data Spark has to scan. Less data scanned = fewer worker nodes needed = lower cloud storage I/O costs.

####Key Learnings & Interview Talking Points
If a hiring manager or tech lead asks, "How do you ensure your Spark code isn't wasting company money?", deploy these answers:

* **Action Minimization**: "I strictly audit my PySpark code for unnecessary actions. Junior developers often pepper their code with `.count()` or `.show()` for debugging, unaware that each call forces Spark to execute a full DAG from scratch. I consolidate transformations and limit my code to a single terminal action, reducing redundant disk I/O."

* **Compute Tiering (Job vs. Interactive)**: "I understand the Databricks pricing model. I do all my exploratory data analysis (EDA) and model prototyping on All-Purpose compute, but when orchestrating the final MLflow training or batch inference pipelines, I strictly configure the workflow to spin up ephemeral Job Clusters to leverage the cheaper DBU rates."

* **`display()` vs. `show()`**: "In Databricks environments, I exclusively use `display()` instead of `.show()`. `.show()` collects data back to the driver node and prints it as plain text, which can cause driver Out-Of-Memory (OOM) errors on large datasets. `display()` safely limits the render payload and provides built-in BI charting without additional compute overhead."